In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
print("Project root added to sys.path")

Project root added to sys.path


In [2]:
import torch
import torch.nn 
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import os

from clients.federated_training import federated_training
from utils.data_partition import dirichlet_partition


torch.manual_seed(42)
np.random.seed(42)

c:\Users\la7tim\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [3]:
transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,))  
    ])
    
train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST('./data', train=False, transform=transform)
    
num_clients = 5
partitions = {
    "dirichlet_iid":      dirichlet_partition(train_dataset, num_clients=num_clients, alpha=1000.0),  # High alpha -> IID-like
    "dirichlet_noniid":   dirichlet_partition(train_dataset, num_clients=num_clients, alpha=0.01),   # Low alpha -> non-IID
    "dirichlet_medium":   dirichlet_partition(train_dataset, num_clients=num_clients, alpha=0.5),   # Medium skew
}

c:\Users\la7tim\Desktop\Internship\FedTinyProp\utils\data_partition.py:6: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  labels = np.array(dataset.targets)


In [4]:
from models.config import get_tinyprop_config
config = get_tinyprop_config("fashionmnist")
print("\nModel Configuration:")
for key, value in config.items():
    print(f"{key}: {value}")

# Initialize model parameters
tinyprop_params = config["tinyprop_params"]
print("\nTinyProp Parameters:")
print(f"S_min: {tinyprop_params.S_min}")
print(f"S_max: {tinyprop_params.S_max}")
print(f"zeta: {tinyprop_params.zeta}")
print(f"number_of_layers: {tinyprop_params.number_of_layers}")


Model Configuration:
tinyprop_params: <models.tinyProp.TinyPropParams object at 0x00000110516D4910>
skip_threshold: 2.5
full_flops_per_batch: 1000000.0
optimizer: {'type': 'sgd', 'lr': 0.001, 'momentum': 0.9}

TinyProp Parameters:
S_min: 0.05
S_max: 0.5
zeta: 0.25
number_of_layers: 2


In [5]:
from clients.aggregators import sparse_fedavg_aggregate
# First, let's create a function to run training for each partition
def train_and_analyze_partition(partition_name, client_datasets, tinyprop_params):
    print(f"\nTraining on partition: {partition_name.upper()}")
    
    # Run training
    model, accuracy_list, flops_list, mem_list, comm_list, sparsity_list, \
    avg_grad_norm_list, avg_phi_list, skipped_batches_list, \
    effective_compute_ratio_list, client_eval_history, compression_ratio_list, \
    history = federated_training(
        client_datasets=client_datasets,
        model_name='fashionmnist',
        testset=test_dataset,
        tinyprop_params=tinyprop_params,
        aggregator_fn=sparse_fedavg_aggregate,
        rounds=100,
        device="cuda" if torch.cuda.is_available() else "cpu",
        local_epochs=1,
        early_stopping_patience=100,
        early_stopping_delta=0.001,
        csv_log_path=f'results/fashionmnist_{partition_name}_training_log.csv',
        initial_sparsity=tinyprop_params.S_min,
        target_sparsity=tinyprop_params.S_max,
        energy_budget=1000
    )
    
    return model, history

partition_results = {}
for strategy_name, client_datasets in partitions.items():
    print(f"\nStarting training for {strategy_name.upper()} partition...")
    try:
        model, history = train_and_analyze_partition(strategy_name, client_datasets, tinyprop_params)
        partition_results[strategy_name] = {
            'model': model,
            'history': history
        }
        print(f"Completed training for {strategy_name.upper()}")
    except Exception as e:
        print(f"Error in {strategy_name.upper()}: {str(e)}")
        continue



Starting training for DIRICHLET_IID partition...

Training on partition: DIRICHLET_IID

Round 1/100

[Client Debug] Starting training for 1 epochs with batch size 32

Epoch 1/1
  - Average Loss: 2.0140
  - Accuracy: 57.11%
  - Smoothed Phi: 1.0000
  - Skipped Batches: 4
  - Current Sparsity: 0.0535
  - Average Gradient Norm: 31.1044
  - Peak Memory: 830.5MB

[Client Debug] Starting training for 1 epochs with batch size 32

Epoch 1/1
  - Average Loss: 2.3263
  - Accuracy: 56.11%
  - Smoothed Phi: 1.0000
  - Skipped Batches: 3
  - Current Sparsity: 0.0541
  - Average Gradient Norm: 32.8267
  - Peak Memory: 856.8MB

[Client Debug] Starting training for 1 epochs with batch size 32

Epoch 1/1
  - Average Loss: 1.9918
  - Accuracy: 55.56%
  - Smoothed Phi: 1.0000
  - Skipped Batches: 5
  - Current Sparsity: 0.0539
  - Average Gradient Norm: 32.3695
  - Peak Memory: 875.4MB

[Client Debug] Starting training for 1 epochs with batch size 32

Epoch 1/1
  - Average Loss: 2.3162
  - Accuracy: 56.

KeyboardInterrupt: 